In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your actual folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_J.xpt"
csv_file = DATA_DIR / "ALQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2017-2018 ALQ_J XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ111",
    "ALQ121",
    "ALQ130",
    "ALQ142",
    "ALQ270",
    "ALQ280",
    "ALQ290",
    "ALQ151",
    "ALQ170"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ALQ_J column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 5533:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 5,533 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached ALQ_J.xpt:
#
#     string variables             = NONE
#     genuine decimal variables    = NONE
#
#
# All ten released fields contain only whole-number:
#
#     respondent identifier
#     response categories
#     counts
#     special codes
#
#
# This is also consistent with the 2017-2018 redesign:
#
# ALQ121 and ALQ142 are coded frequency categories
# instead of the older quantity/unit structure.
#
#
# Therefore all ten can safely become nullable Int64
# AFTER:
#
#     1. tiny-zero repair
#     2. proof that every value is mathematically whole
#
#
# IMPORTANT:
#
# This applies ONLY to ALQ_J.
#
# Genuine Dietary / Laboratory / Examination decimal
# variables in other files must remain exactly AS IS.
#

STRING_COLUMNS = []

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 10:

    raise ValueError(
        "Expected exactly 10 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# NEVER globally replace it without first validating:
#
#     variable
#     count
#     codebook meaning
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 ALQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{ordinary_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached ALQ_J.xpt:
#
#     ALQ121    1,049
#     ALQ142    2,051
#     ALQ270      766
#     ALQ280      917
#     ALQ290      269
#     ALQ170    2,562
#
# TOTAL        7,614
#
#
# CDC explicitly reports zero categories for:
#
# ALQ121 = 0
#     Never in last year
#     Count = 1,049
#
# ALQ142 = 0
#     Never in last year
#     Count = 2,051
#
# ALQ270 = 0
#     Never in last year
#     Count = 766
#
# ALQ280 = 0
#     Never in last year
#     Count = 917
#
# ALQ290 = 0
#     Never in last year
#     Count = 269
#
#
# ALQ170:
#
#     CDC publishes 0-20 as a combined valid range.
#     The exact 2,562 zero count is attachment-specific.
#

EXPECTED_TINY_COUNTS = {
    "ALQ121": 1049,
    "ALQ142": 2051,
    "ALQ270": 766,
    "ALQ280": 917,
    "ALQ290": 269,
    "ALQ170": 2562
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated ALQ_J attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 7614:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 6:

    raise ValueError(
        "Expected tiny values in exactly 6 variables."
    )


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 7,614-value tiny-zero pattern "
    "across 6 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 7614:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df is the numerical fidelity reference.
#
# The ONLY allowed numerical changes are:
#
#     validated tiny -> zero
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY diagnostically.
#
# It NEVER overwrites dataframe values.
#

print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    print(
        f"{col}: "
        f"{count:,} fractional observations"
    )


    if count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables found.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in ALQ_J."
)


# ============================================================
# 11. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 12. PROVE ALL 10 VARIABLES ARE WHOLE-NUMBER
# ============================================================

whole_number_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        whole_number_failures.append(
            col
        )


if whole_number_failures:

    raise ValueError(
        "Unexpected fractional values found in:\n"
        + str(
            whole_number_failures
        )
    )


print(
    "PASS: All 10 ALQ_J variables are "
    "mathematically whole-number."
)


# ============================================================
# 13. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() here.
#
# We already proved every populated value is whole-number.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 10 fields converted to nullable Int64 "
    "without rounding."
)


# ============================================================
# 14. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC/attachment."
    )


# ============================================================
# 15. CDC CHECK - ALQ111
# ============================================================

check_frequency(
    "ALQ111",
    {
        1: 4545,
        2: 585
    },
    403
)


# ============================================================
# 16. CDC CHECK - ALQ121
# ============================================================

EXPECTED_ALQ121 = {
    0: 1049,
    1: 142,
    2: 155,
    3: 273,
    4: 323,
    5: 340,
    6: 606,
    7: 351,
    8: 299,
    9: 476,
    10: 527,
    77: 2,
    99: 2
}


check_frequency(
    "ALQ121",
    EXPECTED_ALQ121,
    988
)


# ============================================================
# 17. ALQ111 -> ALQ121 ROUTING
# ============================================================

#
# CDC:
#
# ALQ111 = 2
#     No lifetime alcohol drink
#     -> End of section
#
#
# Therefore ALQ121 is populated exactly where:
#
#     ALQ111 = 1
#

alq121_expected = (
    df["ALQ111"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alq121_present = (
    df["ALQ121"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


alq121_routing_mismatch = int(
    np.sum(
        alq121_expected
        !=
        alq121_present
    )
)


print(
    "\n--- ALQ111 / ALQ121 ROUTING ---"
)

print(
    "ALQ111=1:",
    int(
        alq121_expected.sum()
    )
)

print(
    "ALQ121 populated:",
    int(
        alq121_present.sum()
    )
)

print(
    "Routing mismatches:",
    alq121_routing_mismatch
)


if int(
    alq121_expected.sum()
) != 4545:

    raise ValueError(
        "Unexpected ALQ121 eligibility count."
    )


if alq121_routing_mismatch != 0:

    raise ValueError(
        "ALQ121 availability differs from ALQ111=1."
    )


print(
    "PASS: ALQ111/ALQ121 routing validated exactly."
)


# ============================================================
# 18. CDC CHECK - ALQ130
# ============================================================

#
# Average number drinks/day on drinking days:
#
#     1-13  = 3,470
#     15    = 19 (15 or more drinks)
#     777   = 1
#     999   = 5
#     Missing = 2,038
#

alq130_range = int(
    df["ALQ130"]
    .between(
        1,
        13
    )
    .sum()
)


alq130_15 = int(
    df["ALQ130"]
    .eq(15)
    .sum()
)


alq130_777 = int(
    df["ALQ130"]
    .eq(777)
    .sum()
)


alq130_999 = int(
    df["ALQ130"]
    .eq(999)
    .sum()
)


alq130_missing = int(
    df["ALQ130"]
    .isna()
    .sum()
)


print(
    "\n--- ALQ130 ---"
)

print(
    "1-13:",
    f"{alq130_range:,}"
)

print(
    "15 or more:",
    alq130_15
)

print(
    "777:",
    alq130_777
)

print(
    "999:",
    alq130_999
)

print(
    "Missing:",
    f"{alq130_missing:,}"
)


if alq130_range != 3470:

    raise ValueError(
        "ALQ130 1-13 count differs."
    )


if alq130_15 != 19:

    raise ValueError(
        "ALQ130 15+ count differs."
    )


if alq130_777 != 1:

    raise ValueError(
        "ALQ130 refused count differs."
    )


if alq130_999 != 5:

    raise ValueError(
        "ALQ130 don't-know count differs."
    )


if alq130_missing != 2038:

    raise ValueError(
        "ALQ130 missing count differs."
    )


print(
    "PASS: ALQ130 matches CDC."
)


# ============================================================
# 19. CDC CHECK - ALQ142
# ============================================================

EXPECTED_ALQ142 = {
    0: 2051,
    1: 34,
    2: 26,
    3: 62,
    4: 103,
    5: 105,
    6: 214,
    7: 142,
    8: 96,
    9: 235,
    10: 418,
    77: 2,
    99: 7
}


check_frequency(
    "ALQ142",
    EXPECTED_ALQ142,
    2038
)


# ============================================================
# 20. CDC CHECK - ALQ270
# ============================================================

EXPECTED_ALQ270 = {
    0: 766,
    1: 12,
    2: 18,
    3: 21,
    4: 37,
    5: 60,
    6: 56,
    7: 85,
    8: 37,
    9: 78,
    10: 259,
    77: 2,
    99: 8
}


check_frequency(
    "ALQ270",
    EXPECTED_ALQ270,
    4094
)


# ============================================================
# 21. CDC CHECK - ALQ280
# ============================================================

EXPECTED_ALQ280 = {
    0: 917,
    1: 9,
    2: 16,
    3: 17,
    4: 28,
    5: 44,
    6: 46,
    7: 55,
    8: 38,
    9: 72,
    10: 190,
    77: 2,
    99: 5
}


check_frequency(
    "ALQ280",
    EXPECTED_ALQ280,
    4094
)


# ============================================================
# 22. CDC CHECK - ALQ290
# ============================================================

EXPECTED_ALQ290 = {
    0: 269,
    1: 2,
    2: 6,
    3: 8,
    4: 14,
    5: 29,
    6: 27,
    7: 25,
    8: 19,
    9: 29,
    10: 89,
    77: 1,
    99: 4
}


check_frequency(
    "ALQ290",
    EXPECTED_ALQ290,
    5011
)


# ============================================================
# 23. ALQ280 -> ALQ290 ROUTING CHECK
# ============================================================

#
# CDC skip:
#
#     ALQ280 = 0
#         -> ALQ151
#
# Therefore ALQ290 is populated exactly when
# ALQ280 is populated and nonzero.
#

alq290_expected = (
    (
        df["ALQ280"]
        .notna()
    )
    &
    (
        ~df["ALQ280"]
        .eq(0)
    )
)


alq290_expected = (
    alq290_expected
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


alq290_present = (
    df["ALQ290"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


alq290_routing_mismatch = int(
    np.sum(
        alq290_expected
        !=
        alq290_present
    )
)


print(
    "\n--- ALQ280 / ALQ290 ROUTING ---"
)

print(
    "Expected ALQ290:",
    int(
        alq290_expected.sum()
    )
)

print(
    "ALQ290 populated:",
    int(
        alq290_present.sum()
    )
)

print(
    "Routing mismatches:",
    alq290_routing_mismatch
)


if int(
    alq290_expected.sum()
) != 522:

    raise ValueError(
        "Unexpected ALQ290 eligibility count."
    )


if alq290_routing_mismatch != 0:

    raise ValueError(
        "ALQ290 availability differs "
        "from ALQ280 routing."
    )


print(
    "PASS: ALQ280/ALQ290 routing validated."
)


# ============================================================
# 24. CDC CHECK - ALQ151
# ============================================================

check_frequency(
    "ALQ151",
    {
        1: 680,
        2: 3857,
        7: 1,
        9: 6
    },
    989
)


# ============================================================
# 25. CDC / ATTACHMENT CHECK - ALQ170
# ============================================================

#
# ALQ170 is DIFFERENT from the other ALQ variables.
#
# CDC notes it was asked at the end of the
# DIETARY interview.
#
# Therefore:
#
#     DO NOT require ALQ170 missingness to perfectly
#     match the main ALQ questionnaire fields.
#
#
# CDC publishes:
#
#     0-20 = 3,411
#     30   = 41
#     777  = 18
#     999  = 17
#     Missing = 2,046
#
#
# Attached XPT exact zero count:
#
#     ALQ170=0 = 2,562
#

alq170_range = int(
    df["ALQ170"]
    .between(
        0,
        20
    )
    .sum()
)


alq170_zero = int(
    df["ALQ170"]
    .eq(0)
    .sum()
)


alq170_30 = int(
    df["ALQ170"]
    .eq(30)
    .sum()
)


alq170_777 = int(
    df["ALQ170"]
    .eq(777)
    .sum()
)


alq170_999 = int(
    df["ALQ170"]
    .eq(999)
    .sum()
)


alq170_missing = int(
    df["ALQ170"]
    .isna()
    .sum()
)


print(
    "\n--- ALQ170 ---"
)

print(
    "0-20:",
    f"{alq170_range:,}"
)

print(
    "Exact zero count:",
    f"{alq170_zero:,}"
)

print(
    "30:",
    alq170_30
)

print(
    "777:",
    alq170_777
)

print(
    "999:",
    alq170_999
)

print(
    "Missing:",
    f"{alq170_missing:,}"
)


if alq170_range != 3411:

    raise ValueError(
        "ALQ170 0-20 range count differs."
    )


if alq170_zero != 2562:

    raise ValueError(
        "ALQ170 exact zero count differs "
        "from attached XPT."
    )


if alq170_30 != 41:

    raise ValueError(
        "ALQ170 30 top-category count differs."
    )


if alq170_777 != 18:

    raise ValueError(
        "ALQ170 refused count differs."
    )


if alq170_999 != 17:

    raise ValueError(
        "ALQ170 don't-know count differs."
    )


if alq170_missing != 2046:

    raise ValueError(
        "ALQ170 missing count differs."
    )


print(
    "PASS: ALQ170 CDC range and exact "
    "attachment zero count validated."
)


# ============================================================
# 26. IMPORTANT ALQ170 ROUTING NOTE
# ============================================================

#
# DO NOT impose a strict:
#
#     ALQ121 -> ALQ170
#
# row-by-row requirement.
#
# CDC explicitly notes:
#
# ALQ170 was asked in the dietary interview.
#
# Therefore some respondents have:
#
#     ALQ170 but not other ALQ data
#
# or:
#
#     ALQ data but no ALQ170
#
#
# This is expected behavior, not a fidelity failure.
#

print(
    "\nPASS: No inappropriate cross-component "
    "routing constraint imposed on ALQ170."
)


# ============================================================
# 27. VALID RESPONSE CODE CHECK
# ============================================================

CATEGORY_CODES = {
    "ALQ111": {
        1, 2
    },

    "ALQ121": {
        0, 1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        77, 99
    },

    "ALQ142": {
        0, 1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        77, 99
    },

    "ALQ270": {
        0, 1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        77, 99
    },

    "ALQ280": {
        0, 1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        77, 99
    },

    "ALQ290": {
        0, 1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        77, 99
    },

    "ALQ151": {
        1, 2, 7, 9
    }
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed in CATEGORY_CODES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    print(
        f"{col}:",
        sorted(observed)
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected categorical response codes."
)


# ============================================================
# 28. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original pandas-decoded XPT had:
#
#     ordinary zero count = 0
#
# Therefore every real zero after restoration must
# exactly correspond to a validated tiny-value position.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 7614:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 7,614 validated restorations."
)


# ============================================================
# 29. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ONLY:
#
#     validated tiny -> zero
#
# changes are allowed.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 10 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 30. STORE MISSING COUNTS BEFORE EXPORT
# ============================================================

expected_missing_counts = {
    col: int(
        expected_df[col]
        .isna()
        .sum()
    )
    for col in EXPECTED_COLUMNS
}


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# ALQ_J itself has no genuine decimal variables.
#
# This export method remains safe for your broader
# NHANES workflow where genuine decimals must remain
# exactly AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 32. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 33. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 10 ALQ_J variables."
)


# ============================================================
# 34. MISSING-VALUE FIDELITY
# ============================================================

missing_position_mismatches = 0

missing_count_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    actual_missing_count = int(
        roundtrip_df[col]
        .isna()
        .sum()
    )


    if (
        actual_missing_count
        !=
        expected_missing_counts[col]
    ):

        missing_count_failures.append(
            col
        )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


if missing_count_failures:

    raise ValueError(
        "Missing counts changed in:\n"
        + str(
            missing_count_failures
        )
    )


print(
    "PASS: Missing positions/counts preserved exactly."
)


# ============================================================
# 35. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 36. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 37. RAW CSV .0 CHECK
# ============================================================

#
# ALQ_J contains no genuine decimal columns.
#
# Therefore checking all ten fields for unnecessary
# ".0" is safe for THIS file.
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- RAW CSV .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any ALQ_J field."
)


# ============================================================
# 38. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 7,614 validated zero values."
)


# ============================================================
# 39. FINAL KEY FREQUENCY CHECKS
# ============================================================

FINAL_FREQUENCIES = {

    "ALQ111": {
        1: 4545,
        2: 585
    },

    "ALQ121": EXPECTED_ALQ121,

    "ALQ142": EXPECTED_ALQ142,

    "ALQ270": EXPECTED_ALQ270,

    "ALQ280": EXPECTED_ALQ280,

    "ALQ290": EXPECTED_ALQ290,

    "ALQ151": {
        1: 680,
        2: 3857,
        7: 1,
        9: 6
    }
}


print(
    "\n--- FINAL FREQUENCY CHECKS ---"
)


for col, expected_freq in FINAL_FREQUENCIES.items():

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    actual_freq = (
        values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 40. FINAL ALQ170 RANGE CHECK
# ============================================================

final_alq170 = pd.to_numeric(
    raw_csv_df["ALQ170"]
    .where(
        raw_csv_df["ALQ170"] != "",
        np.nan
    ),
    errors="coerce"
)


if int(
    final_alq170
    .between(
        0,
        20
    )
    .sum()
) != 3411:

    raise ValueError(
        "Final ALQ170 0-20 range changed."
    )


if int(
    final_alq170
    .eq(0)
    .sum()
) != 2562:

    raise ValueError(
        "Final ALQ170 zero count changed."
    )


if int(
    final_alq170
    .eq(30)
    .sum()
) != 41:

    raise ValueError(
        "Final ALQ170 30 count changed."
    )


if int(
    final_alq170
    .eq(777)
    .sum()
) != 18:

    raise ValueError(
        "Final ALQ170 refused count changed."
    )


if int(
    final_alq170
    .eq(999)
    .sum()
) != 17:

    raise ValueError(
        "Final ALQ170 don't-know count changed."
    )


print(
    "PASS: Final ALQ170 distribution preserved."
)


# ============================================================
# 41. FINAL ROUTING CHECKS
# ============================================================

final_alq111 = pd.to_numeric(
    raw_csv_df["ALQ111"]
    .where(
        raw_csv_df["ALQ111"] != "",
        np.nan
    ),
    errors="coerce"
)


final_alq121_present = (
    raw_csv_df["ALQ121"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


expected_alq121_present = (
    final_alq111
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_alq121_mismatch = int(
    np.sum(
        expected_alq121_present
        !=
        final_alq121_present
    )
)


final_alq280 = pd.to_numeric(
    raw_csv_df["ALQ280"]
    .where(
        raw_csv_df["ALQ280"] != "",
        np.nan
    ),
    errors="coerce"
)


expected_alq290_present = (
    (
        final_alq280.notna()
    )
    &
    (
        ~final_alq280.eq(0)
    )
)


expected_alq290_present = (
    expected_alq290_present
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


actual_alq290_present = (
    raw_csv_df["ALQ290"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_alq290_mismatch = int(
    np.sum(
        expected_alq290_present
        !=
        actual_alq290_present
    )
)


print(
    "\n--- FINAL ROUTING CHECKS ---"
)

print(
    "ALQ111/ALQ121 mismatches:",
    final_alq121_mismatch
)

print(
    "ALQ280/ALQ290 mismatches:",
    final_alq290_mismatch
)


if final_alq121_mismatch != 0:

    raise ValueError(
        "Final ALQ111/ALQ121 routing changed."
    )


if final_alq290_mismatch != 0:

    raise ValueError(
        "Final ALQ280/ALQ290 routing changed."
    )


print(
    "PASS: Final within-component routing preserved."
)


# ============================================================
# 42. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 43. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 44. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ALQ_J CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2017-2018"
)


print(
    "Component: Questionnaire - Alcohol Use"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "ALQ121 zeros:",
    f"{int(df['ALQ121'].eq(0).sum()):,}"
)


print(
    "ALQ142 zeros:",
    f"{int(df['ALQ142'].eq(0).sum()):,}"
)


print(
    "ALQ270 zeros:",
    f"{int(df['ALQ270'].eq(0).sum()):,}"
)


print(
    "ALQ280 zeros:",
    f"{int(df['ALQ280'].eq(0).sum()):,}"
)


print(
    "ALQ290 zeros:",
    f"{int(df['ALQ290'].eq(0).sum()):,}"
)


print(
    "ALQ170 zeros:",
    f"{int(df['ALQ170'].eq(0).sum()):,}"
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "ALQ111/ALQ121 routing mismatches:",
    alq121_routing_mismatch
)


print(
    "ALQ280/ALQ290 routing mismatches:",
    alq290_routing_mismatch
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nIMPORTANT ALQ170 NOTE:"
)


print(
    "ALQ170 was collected at the end of the "
    "dietary interview."
)


print(
    "Therefore ALQ170 is NOT forced to have the same "
    "missingness/routing pattern as the other ALQ fields."
)


print(
    "\nFINAL RULE:"
)


print(
    "Only the 7,614 validated tiny XPORT values "
    "are changed to real numeric zero."
)


print(
    "All 10 ALQ_J fields are proven whole-number "
    "before nullable Int64 conversion."
)


print(
    "No df.round() and no float_format are used."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "values remain exactly AS IS."
)

NHANES 2017-2018 ALQ_J XPT -> CSV
Rows: 5,533
Columns: 10
PASS: Structure = 5,533 rows x 10 columns.

--- FIELD TYPE PLAN ---
Character fields: []
Genuine decimal fields: []
Whole-number/code fields: ['SEQN', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ142', 'ALQ270', 'ALQ280', 'ALQ290', 'ALQ151', 'ALQ170']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 7,614
Variables containing tiny artifacts: 6

Tiny values by variable:
ALQ170    2562
ALQ142    2051
ALQ121    1049
ALQ280     917
ALQ270     766
ALQ290     269

PASS: Exact 7,614-value tiny-zero pattern across 6 variables validated.

--- VALIDATED ZERO RESTORATION ---
ALQ121: 1,049 restored to 0
ALQ142: 2,051 restored to 0
ALQ270: 766 restored to 0
ALQ280: 917 restored to 0
ALQ290: 269 restored to 0
ALQ170: 2,562 restored to 0

PASS: 7,614 validated zero values restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE DETECTION ---
SEQN: 0 fractional observ